# 11 · F.R.I.D.A.Y. opens the archive – RAG three ways

J.A.R.V.I.S. answers every question with confidence, also about a bolt torque he never read. F.R.I.D.A.Y. makes him open the archive first and point to the page. That is **retrieval-augmented generation (RAG)**: find the passages that matter, put them into the prompt, and let the model answer from them.

You build it three times. First **by hand**, so you see every step. Then with **file search**, where Foundry does the steps for you. Then, optionally, with **Azure AI Search**, where you own the index.

| Concept | Meaning |
|---|---|
| **Chunk** | a piece of a document small enough to retrieve and to quote |
| **Embedding** | a vector for a text; similar meaning, similar direction (chapter 01) |
| **Retrieval** | rank all chunks against the question and keep the best `k` (**top-k**) |
| **Augmented prompt** | the rules, the retrieved chunks as numbered sources, then the question |
| **Citation** | which source an answer used: `[1]` by hand, `file_citation` or `url_citation` from a managed tool |
| **Groundedness** | does the answer say only what the retrieved context supports? A judge model scores it |

| Variant | Who chunks | Who embeds | Who retrieves | Needs |
|---|---|---|---|---|
| By hand (section 2) | you | you, with `STARK_EMBEDDING_MODEL` | you, cosine similarity in Python | nothing extra |
| File search (section 3) | Foundry | Foundry | Foundry, the model writes the query | nothing extra |
| Azure AI Search (section 4) | you | you | the search service: keyword, vector or hybrid | `STARK_ENABLE_SEARCH=true`, optional |

Builds on chapter 01 (embeddings), 04 (agents), 05 (file search) and 10 (evaluations). Deep dive: [docs/rag.md](../docs/rag.md).

> ❓ **Not run on a live project yet.** The calls follow Microsoft Learn as of 2026-11. Every outcome that only a live run can confirm is marked ❓.

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_evals import show_scores, show_pass_rates
from stark_rag import (RULES, NOT_FOUND, load_archive, split_by_heading, split_fixed, cosine_similarity, retrieve,
                       grounded_prompt, cited_labels, rank_of, yes_no, file_search_queries, file_search_hits, citations,
                       cited_sources, show_chunks, show_hits, show_rag_turn, search_ready, search_index_client,
                       search_client, archive_index, search_documents, search_hits, eval_row, upload_rows,
                       rows_dataset_name)

from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool

project_openai = openai_client()     # the project's OpenAI client: responses, vector stores, evals
rag_rows = {"by hand": {}, "file search": {}, "AI Search": {}}   # question -> evaluation row, filled by the cases
to_clean = {"agents": [], "evals": [], "vector_stores": [], "datasets": [], "indexes": []}

## 1 · The archive and the baseline
The archive is five Markdown files in [data/archive](../data/archive): the [maintenance manual](../data/archive/arc-reactor-maintenance.md), the [suit registry](../data/archive/suit-registry.md), the [procurement policy](../data/archive/procurement-policy.md), the [safety rules](../data/archive/workshop-safety.md) and the [incident log](../data/archive/incident-log.md). No model was trained on them.

Four questions run through the whole chapter:

| Name | Question | Where the answer is |
|---|---|---|
| `torque_question` | What torque do the arc reactor housing bolts need? | one section: maintenance manual, *Housing bolts* (42 Nm) |
| `approval_question` | Who has to sign off before I buy something expensive? | one section: procurement policy, *Purchase approvals*. It shares no keyword with the question |
| `leak_question` | Why did the coolant leak in August, and what is the correct value? | two sections in two files: the incident log (24 Nm) and the maintenance manual (42 Nm) |
| `wifi_question` | What is the workshop Wi-Fi password? | nowhere |

`load_archive()` ([stark_rag.py](stark_rag.py)) reads the files.

**Expected:** five files, about 5,600 characters in total.

In [ ]:
archive = load_archive()             # file name -> text
for name, text in archive.items():
    print(f"{name:28} {len(text):5} characters")

torque_question = "What torque do the arc reactor housing bolts need?"
approval_question = "Who has to sign off before I buy something expensive?"
leak_question = "Why did the coolant leak in August, and what is the correct value?"
wifi_question = "What is the workshop Wi-Fi password?"

### Case A · No retrieval

**Goal:** the baseline. The model gets the question and nothing else, so it can only guess or decline.

| Input | Value |
|---|---|
| question | `torque_question` |
| model | `STARK_CHAT_MODEL`, no instructions, no tools |
| sources | none |

**Expected:** ❓ the model cannot know Stark's manual. It either says so or names a plausible number. `says 42 Nm` should be `no`; a `yes` would be a lucky guess, not knowledge.

In [ ]:
baseline_reply = project_openai.responses.create(model=model("chat"), input=torque_question)
show_rag_turn(torque_question, answer_text(baseline_reply), {"says 42 Nm": yes_no("42" in final_message(baseline_reply), expected=False)},
              to=model("chat"))

## 2 · RAG by hand
Five steps: **chunk** the documents, **embed** the chunks, **retrieve** the chunks closest to the question, **augment** the prompt with them, **answer**. Steps 1 and 2 happen once per archive (indexing). The rest happens for every question.

### Step 1 · Chunk

**Goal:** pieces that each hold one topic. `split_by_heading()` ([stark_rag.py](stark_rag.py)) makes one chunk per `## ` section and puts "Title › Heading" on its first line, so a chunk still makes sense alone.

**Expected:** 25 chunks, five per file, each between 140 and 380 characters. The one to remember is `arc-reactor-maintenance-2` (*Housing bolts*): the only place that says 42 Nm.

In [ ]:
chunks = [chunk for name, text in archive.items() for chunk in split_by_heading(text, name)]
show_chunks(chunks)
print(chunks[1]["text"])

### Step 2 · Embed

**Goal:** one vector per chunk, so "close in meaning" becomes "close in direction". This is the call from chapter 01, now with 25 texts in one request. Embeddings use the account endpoint (`embeddings_client()`), not the project route.

**Expected:** `25 vectors of 1536 dimensions` with the default `text-embedding-3-small`.

In [ ]:
embedding_result = embeddings_client().embeddings.create(model=model("embedding"), input=[chunk["text"] for chunk in chunks])
chunk_vectors = [row.embedding for row in embedding_result.data]
print(len(chunk_vectors), "vectors of", len(chunk_vectors[0]), "dimensions")

### Step 3 · Retrieve

**Goal:** the retrieval itself, written out once: embed the question, compare it with every chunk, sort, keep three. A vector database does the same with an index that avoids comparing everything.

| Input | Value |
|---|---|
| question | `torque_question` |
| `k` | 3 |

`show_hits()` shows the ranked chunks and folds their text. From here on `retrieve(question, chunks, chunk_vectors, k)` runs these three lines.

**Expected:** rank 1 is `arc-reactor-maintenance.md › Housing bolts` ❓. The score is a cosine similarity, typically between 0.3 and 0.7 for this model ❓. The other two ranks are sections that also talk about bolts or the reactor.

In [ ]:
question_vector = embeddings_client().embeddings.create(model=model("embedding"), input=[torque_question]).data[0].embedding
scored = [(cosine_similarity(question_vector, vector), chunk) for chunk, vector in zip(chunks, chunk_vectors)]
torque_hits = sorted(scored, key=lambda hit: hit[0], reverse=True)[:3]

show_hits(torque_question, torque_hits, {
    "rank of Housing bolts": f"{rank_of(torque_hits, 'arc-reactor-maintenance.md', 'Housing bolts')} (expected 1)"})

### Case B · Augment and answer

**Goal:** the same question as case A, now with the three chunks in the prompt. Only the prompt changed: same model, no tools.

| Input | Value |
|---|---|
| question | `torque_question` |
| sources | `torque_hits` from step 3, numbered `[1]` to `[3]` |
| rules | `RULES`: answer only from the sources, cite them as `[n]`, say "That is not in the archive." when they don't hold the answer |

The cell builds the prompt once by hand. Later cases call `grounded_prompt(question, hits)`, which builds the same text. The prompt looks like this:

```text
Answer the question using only the sources below. Cite each source you use as [1], [2], ... If the sources do not
contain the answer, say 'That is not in the archive.' and do not guess.

Sources:
[1] arc-reactor-maintenance.md › Housing bolts
Arc Reactor Maintenance Manual › Housing bolts
The housing is closed with eight titanium bolts. Housing bolts are tightened to 42 Nm in a star pattern, ...

[2] ...

Question: What torque do the arc reactor housing bolts need?
```

`show_rag_turn()` shows the prompt (sources folded), the hits, the whole answer and this check table:

| | |
|---|---|
| says 42 Nm | yes (expected yes) ✅ |
| cites | arc-reactor-maintenance.md › Housing bolts |

**Expected:** 42 Nm, with a `[1]` that points to *Housing bolts* ❓ (the number depends on the rank in step 3).

In [ ]:
sources = "\n\n".join(f"[{number}] {chunk['source']} › {chunk['heading']}\n{chunk['text']}"
                      for number, (_, chunk) in enumerate(torque_hits, 1))
torque_prompt = f"{RULES}\n\nSources:\n{sources}\n\nQuestion: {torque_question}"

torque_reply = project_openai.responses.create(model=model("chat"), input=torque_prompt)
torque_answer = answer_text(torque_reply)
rag_rows["by hand"][torque_question] = eval_row(torque_question, torque_hits, torque_answer)
show_rag_turn(torque_prompt, torque_answer, {
    "says 42 Nm": yes_no("42" in torque_answer, expected=True),
    "cites": ", ".join(cited_labels(torque_answer, torque_hits)) or "nothing"}, to=model("chat"), hits=torque_hits)

### Case C · The answer is not in the archive

**Goal:** retrieval always returns `k` chunks, also when none fits. The rule in the prompt has to stop the model from answering anyway.

| Input | Value |
|---|---|
| question | `wifi_question` |
| `k` | 3 |

**Expected:** three hits with lower scores than in step 3 ❓, none about Wi-Fi. The answer is "That is not in the archive." and no password ❓.

In [ ]:
wifi_hits = retrieve(wifi_question, chunks, chunk_vectors, k=3)
wifi_prompt = grounded_prompt(question=wifi_question, hits=wifi_hits)

wifi_answer = answer_text(project_openai.responses.create(model=model("chat"), input=wifi_prompt))
rag_rows["by hand"][wifi_question] = eval_row(wifi_question, wifi_hits, wifi_answer)
show_rag_turn(wifi_prompt, wifi_answer, {
    "says it is not in the archive": yes_no(NOT_FOUND.lower() in wifi_answer.lower(), expected=True),
    "top score": f"{wifi_hits[0][0]:.3f} (step 3: {torque_hits[0][0]:.3f})"}, to=model("chat"), hits=wifi_hits)

### Case D · Change k only

**Goal:** a question whose answer sits in two chunks. With `k=1` the model sees half of it.

| Input | Step 1 | Step 2 |
|---|---|---|
| question | `leak_question` | `leak_question` |
| `k` | 1 | 3 |

**Step 1 expected:** one hit ❓, probably `incident-log.md › 2026-08-14 · Coolant leak`. The answer names the cause (24 Nm) but not the correct value, because the incident log only says "the value in the maintenance manual".

In [ ]:
leak_hits_k1 = retrieve(leak_question, chunks, chunk_vectors, k=1)
leak_prompt_k1 = grounded_prompt(question=leak_question, hits=leak_hits_k1)

leak_answer_k1 = answer_text(project_openai.responses.create(model=model("chat"), input=leak_prompt_k1))
show_rag_turn(leak_prompt_k1, leak_answer_k1, {
    "names 24 Nm (the cause)": yes_no("24" in leak_answer_k1), "names 42 Nm (the correct value)": yes_no("42" in leak_answer_k1)},
    to=model("chat"), hits=leak_hits_k1)

**Step 2 expected:** three hits that include both the incident and *Housing bolts* ❓. The answer names 24 Nm and 42 Nm and cites two sources.

> 💡 A larger `k` finds more, costs more tokens and adds text that does not matter. Three to five is a common start; measure before you raise it.

In [ ]:
leak_hits = retrieve(leak_question, chunks, chunk_vectors, k=3)
leak_prompt = grounded_prompt(question=leak_question, hits=leak_hits)

leak_answer = answer_text(project_openai.responses.create(model=model("chat"), input=leak_prompt))
rag_rows["by hand"][leak_question] = eval_row(leak_question, leak_hits, leak_answer)
show_rag_turn(leak_prompt, leak_answer, {
    "names 24 Nm (the cause)": yes_no("24" in leak_answer, expected=True),
    "names 42 Nm (the correct value)": yes_no("42" in leak_answer, expected=True),
    "cites": ", ".join(cited_labels(leak_answer, leak_hits)) or "nothing"}, to=model("chat"), hits=leak_hits)

### Case E · Change the chunking only

**Goal:** see what a careless cut does. `split_fixed(text, source, size=200)` cuts every 200 characters, wherever that falls. Same question and `k` as step 3; only the chunks differ.

| Input | Step 3 | This case |
|---|---|---|
| chunks | 25, one per section | 30, 200 characters each |
| question | `torque_question` | `torque_question` |

**Expected:** the cut falls in the middle of the sentence. The chunk with the number starts with "e tightened to 42 Nm in a star pattern" and no longer says what is tightened; "Housing bolts ar" ends the chunk before it. Both facts are computed, so they are the same in every run. ❓ Where the chunk with the number ranks: it may still be in the top 3, lower than in step 3, or gone.

> 💡 This is why chunkers cut at headings or sentences and let chunks overlap. File search (section 3) uses 800 tokens with 400 overlap by default.

In [ ]:
fixed_chunks = [chunk for name, text in archive.items() for chunk in split_fixed(text, name, size=200)]
fixed_vectors = [row.embedding for row in embeddings_client().embeddings.create(
    model=model("embedding"), input=[chunk["text"] for chunk in fixed_chunks]).data]

fixed_hits = retrieve(torque_question, fixed_chunks, fixed_vectors, k=3)
number_chunk = next(chunk for chunk in fixed_chunks if "42 Nm" in chunk["text"])
show_hits(torque_question, fixed_hits, {
    "chunk with 42 Nm": f"`{number_chunk['id']}`, starts with \"{number_chunk['text'][:38]}\"",
    "it names the housing bolts": yes_no("Housing bolts" in number_chunk["text"], expected=False),
    "its rank in the top 3": rank_of(fixed_hits, number_chunk["source"], number_chunk["heading"]) or "not in the top 3"})

## 3 · Managed: file search
Chapter 05 used file search as a tool. Now you know what it does inside: Foundry chunks each uploaded file, embeds the chunks, and stores them in a **vector store**. At question time the model writes a search query, Foundry retrieves, and the answer carries `file_citation` annotations.

| Step | By hand (section 2) | File search |
|---|---|---|
| Chunk | `split_by_heading`, one chunk per section | 800 tokens with 400 overlap by default, so each small archive file is one chunk |
| Embed | `STARK_EMBEDDING_MODEL` | a model Foundry manages, not your deployment |
| Retrieve | cosine similarity, top 3 | keyword plus vector search with reranking, up to 20 chunks |
| Query | the question as typed | the model rewrites it |

The defaults are from [Microsoft Learn](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/file-search). You can set `chunking_strategy` when you add a file ❓ (in the REST reference, not tried through `upload_and_poll` here).

### Step 1 · Vector store and agent

**Goal:** the archive in a vector store, and an agent that may only answer from it.

Each run of the next cell creates a new vector store named `stark-archive` and a new version of `friday-archive-files`. The clean-up deletes what the last run created.

**Expected:** `5 files in vector store vs_...` and the agent's version.

In [ ]:
archive_store = project_openai.vector_stores.create(name="stark-archive")
to_clean["vector_stores"].append(archive_store.id)
for path in sorted((ROOT / "data" / "archive").glob("*.md")):
    with open(path, "rb") as f:
        project_openai.vector_stores.files.upload_and_poll(vector_store_id=archive_store.id, file=f)
print(len(archive), "files in vector store", archive_store.id)

FILES_AGENT_NAME = "friday-archive-files"
ARCHIVE_INSTRUCTIONS = ("You are F.R.I.D.A.Y., the archivist. Answer only from the archive (use your search tool). "
                        f"If the archive does not contain the answer, say '{NOT_FOUND}' and do not guess.")
files_agent = project().agents.create_version(agent_name=FILES_AGENT_NAME, definition=PromptAgentDefinition(
    model=model("chat"), instructions=ARCHIVE_INSTRUCTIONS, tools=[FileSearchTool(vector_store_ids=[archive_store.id])]))
to_clean["agents"].append(FILES_AGENT_NAME)
print("agent", files_agent.name, "version", files_agent.version)

### Case F · The same question, and what was retrieved

**Goal:** look inside the managed retrieval. By default a response only says that a `file_search_call` happened. `include=["file_search_call.results"]` adds the retrieved chunks:

```python
project_openai.responses.create(input=torque_question, extra_body=agent_ref(files_agent),
                                include=["file_search_call.results"])
```

| Input | Value |
|---|---|
| question | `torque_question`, as typed, no sources in the prompt |
| agent | `friday-archive-files` |

What `show_rag_turn()` shows, and where it comes from ([stark_rag.py](stark_rag.py)):

| Part | Read by | From |
|---|---|---|
| search queries | `file_search_queries()` | `queries` of the `file_search_call` item |
| retrieved chunks | `file_search_hits()` | `results` of that item: `filename`, `score`, `text` |
| citations | `citations()` | `file_citation` annotations of the answer |

**Expected:** 42 Nm, and a `file_citation` that names `arc-reactor-maintenance.md` ❓. The top result is that file ❓. ❓ Whether `include` returns results when the request goes to an agent (`agent_reference`) was not tried: if "Retrieved" says *nothing retrieved* while the answer is right, it does not.

In [ ]:
files_torque_reply = project_openai.responses.create(input=torque_question, extra_body=agent_ref(files_agent),
                                                     include=["file_search_call.results"])
files_torque_hits = file_search_hits(files_torque_reply)
rag_rows["file search"][torque_question] = eval_row(torque_question, files_torque_hits, answer_text(files_torque_reply))
show_rag_turn(torque_question, answer_text(files_torque_reply), {
    "says 42 Nm": yes_no("42" in final_message(files_torque_reply), expected=True),
    "top result": files_torque_hits[0][1]["source"] if files_torque_hits else "no results returned",
    "cited files": ", ".join(cited_sources(files_torque_reply)) or "none"},
    to=FILES_AGENT_NAME, hits=files_torque_hits, queries=file_search_queries(files_torque_reply), cites=citations(files_torque_reply))

### Case G · The answer is not in the archive

**Goal:** case C again, with the managed retriever.

| Input | Value |
|---|---|
| question | `wifi_question` |
| agent | `friday-archive-files` |

**Expected:** "That is not in the archive." ❓ and no citations ❓. File search may still return results: like top-k by hand, it returns the closest chunks it has.

In [ ]:
files_wifi_reply = project_openai.responses.create(input=wifi_question, extra_body=agent_ref(files_agent),
                                                   include=["file_search_call.results"])
files_wifi_hits = file_search_hits(files_wifi_reply)
rag_rows["file search"][wifi_question] = eval_row(wifi_question, files_wifi_hits, answer_text(files_wifi_reply))
show_rag_turn(wifi_question, answer_text(files_wifi_reply), {
    "says it is not in the archive": yes_no(NOT_FOUND.lower() in final_message(files_wifi_reply).lower(), expected=True),
    "results returned": len(files_wifi_hits), "cited files": ", ".join(cited_sources(files_wifi_reply)) or "none"},
    to=FILES_AGENT_NAME, hits=files_wifi_hits, queries=file_search_queries(files_wifi_reply), cites=citations(files_wifi_reply))

### Case H · An answer from two files

**Goal:** case D step 2 again. By hand you had to raise `k`. Here the model decides what to search for, and may search more than once.

| Input | Value |
|---|---|
| question | `leak_question` |
| agent | `friday-archive-files` |

**Expected:** 24 Nm and 42 Nm ❓, with citations for `incident-log.md` and `arc-reactor-maintenance.md` ❓. Look at the search queries: ❓ one query or several.

In [ ]:
files_leak_reply = project_openai.responses.create(input=leak_question, extra_body=agent_ref(files_agent),
                                                   include=["file_search_call.results"])
files_leak_hits = file_search_hits(files_leak_reply)
rag_rows["file search"][leak_question] = eval_row(leak_question, files_leak_hits, answer_text(files_leak_reply))
show_rag_turn(leak_question, answer_text(files_leak_reply), {
    "names 24 Nm (the cause)": yes_no("24" in final_message(files_leak_reply), expected=True),
    "names 42 Nm (the correct value)": yes_no("42" in final_message(files_leak_reply), expected=True),
    "cited files": ", ".join(cited_sources(files_leak_reply)) or "none"},
    to=FILES_AGENT_NAME, hits=files_leak_hits, queries=file_search_queries(files_leak_reply), cites=citations(files_leak_reply))

### Step 2 · Search the vector store without a model

**Goal:** retrieval only, as in section 2 step 3, against the managed store: `vector_stores.search(...)`.

**Expected:** ❓ not tried on a Foundry project. The call exists in the OpenAI SDK and the Azure OpenAI preview REST API; the project route may answer with an error. The `try` keeps the notebook running and prints which one you got.

In [ ]:
try:
    store_results = project_openai.vector_stores.search(archive_store.id, query=torque_question, max_num_results=3)
    store_hits = [(result.score, {"source": result.filename, "heading": "",
                                  "text": "\n".join(part.text for part in result.content)}) for result in store_results]
    show_hits(torque_question, store_hits)
except Exception as exc:
    print("vector_stores.search is not available here:", type(exc).__name__, str(exc)[:300])

## 4 · Your own index: Azure AI Search (optional)
File search hides the index. **Azure AI Search** is a search service you own: you define the fields, you upload the chunks, and you choose how to query. It is not part of the default deployment.

> ⚠️ **Optional and billed.** Deploy it with `azd env set STARK_ENABLE_SEARCH true`, then `azd provision` ([search.bicep](../infra/modules/search.bicep)). The Basic tier is billed by the hour for as long as the service exists; see [pricing](https://azure.microsoft.com/pricing/details/search/). Setting the flag back to `false` does not delete it: use `azd down`, or delete the search service in the Azure portal.
>
> Without it, every code cell in this section prints how to enable it and does nothing. Sections 5 and later still run.

`azd provision` creates the service with Entra-only access, gives you and the Foundry identities the search roles, and adds the project connection `stark-search`. `.env` then holds `STARK_SEARCH_ENDPOINT` and `STARK_SEARCH_CONNECTION`.

### Step 1 · Create the index and upload the chunks

**Goal:** the chunks and vectors from section 2 become search documents. Indexing by hand and indexing a search service are the same pipeline; only the store differs.

`archive_index()` ([stark_rag.py](stark_rag.py)) defines the fields, and `search_documents()` fills them:

| Field | Holds | Used for |
|---|---|---|
| `id` | the chunk id, such as `arc-reactor-maintenance-2` | the key |
| `content`, `title` | the chunk text; "file › heading" | keyword search |
| `content_vector` | the embedding, 1536 numbers | vector search |
| `source`, `heading` | the file name and the section | showing a hit |
| `url` | a link to the file on GitHub | the citation of the agent tool (case K) |

Running the cell again updates the same index and the same 25 documents.

**Expected:** `25 documents uploaded to stark-archive`. ❓ A `403` right after `azd provision` means the role assignment has not reached the service yet: wait a few minutes and run the cell again.

In [ ]:
INDEX_NAME = "stark-archive"
ARCHIVE_URL = "https://github.com/nampacx/MS-Foundry-from-0-to-hero/blob/main/data/archive"

if search_ready():
    search_index_client().create_or_update_index(archive_index(INDEX_NAME, dimensions=len(chunk_vectors[0])))
    to_clean["indexes"].append(INDEX_NAME)
    archive_search = search_client(INDEX_NAME)
    uploaded = archive_search.upload_documents(documents=search_documents(chunks, chunk_vectors, base_url=ARCHIVE_URL))
    print(sum(result.succeeded for result in uploaded), "documents uploaded to", INDEX_NAME)

### Case I · Keyword, vector, hybrid

**Goal:** one question, three ways to query, one change per step. `approval_question` asks who has to "sign off" before you "buy something expensive". The section that answers it says "Pepper Potts approves all purchases above 5,000 USD": the same meaning, no shared keyword. The safety rules, however, contain "sign off" and "before".

| Input | Step 1 | Step 2 | Step 3 |
|---|---|---|---|
| question | `approval_question` | `approval_question` | `approval_question` |
| `search_text` | the question | none | the question |
| `vector_queries` | none | the question's embedding | the question's embedding |

**Step 1 expected:** keyword search counts matching words (BM25). ❓ Rank 1 is probably `workshop-safety.md › Test flights`, which has the words but not the answer. *Purchase approvals* ranks low or is missing. Wait a few seconds after step 1 of this section: new documents take a moment to become searchable.

In [ ]:
if search_ready():
    keyword_hits = search_hits(archive_search.search(search_text=approval_question, top=3))
    show_hits(approval_question, keyword_hits, {
        "rank of Purchase approvals": rank_of(keyword_hits, "procurement-policy.md", "Purchase approvals") or "not in the top 3"})

**Step 2 expected:** vector search compares meaning, as in section 2. ❓ Rank 1 is `procurement-policy.md › Purchase approvals`. The ranking should match `retrieve(approval_question, chunks, chunk_vectors)`, because the vectors are the same; the service uses an approximate index (HNSW), which makes no difference at 25 documents.

In [ ]:
if search_ready():
    from azure.search.documents.models import VectorizedQuery
    approval_vector = embeddings_client().embeddings.create(model=model("embedding"), input=[approval_question]).data[0].embedding
    approval_query = VectorizedQuery(vector=approval_vector, k_nearest_neighbors=3, fields="content_vector")

    vector_hits = search_hits(archive_search.search(search_text=None, vector_queries=[approval_query], top=3))
    show_hits(approval_question, vector_hits, {
        "rank of Purchase approvals": rank_of(vector_hits, "procurement-policy.md", "Purchase approvals") or "not in the top 3"})

**Step 3 expected:** hybrid search runs both and merges the two rankings (reciprocal rank fusion). ❓ *Purchase approvals* is in the top 3, and so is the keyword favourite. The scores are small numbers (around 0.03) because they come from ranks, not from similarity: don't compare scores between the three steps.

> 💡 Hybrid is the usual default: vectors find paraphrases, keywords find exact names, part numbers and codes that embeddings blur.

In [ ]:
if search_ready():
    hybrid_hits = search_hits(archive_search.search(search_text=approval_question, vector_queries=[approval_query], top=3))
    show_hits(approval_question, hybrid_hits, {
        "rank of Purchase approvals": rank_of(hybrid_hits, "procurement-policy.md", "Purchase approvals") or "not in the top 3"})

### Case J · Swap the retriever only

**Goal:** case B again with one change: the three chunks come from a hybrid query against the index instead of `retrieve()`. Same rules, same prompt builder, same model.

| Input | Case B | This case |
|---|---|---|
| question | `torque_question` | `torque_question` |
| retriever | cosine similarity in Python | Azure AI Search, hybrid, `top=3` |

**Expected:** 42 Nm with a citation of *Housing bolts* ❓, as in case B.

In [ ]:
if search_ready():
    search_torque_vector = embeddings_client().embeddings.create(model=model("embedding"), input=[torque_question]).data[0].embedding
    search_torque_hits = search_hits(archive_search.search(search_text=torque_question, top=3, vector_queries=[
        VectorizedQuery(vector=search_torque_vector, k_nearest_neighbors=3, fields="content_vector")]))
    search_torque_prompt = grounded_prompt(question=torque_question, hits=search_torque_hits)

    search_torque_answer = answer_text(project_openai.responses.create(model=model("chat"), input=search_torque_prompt))
    rag_rows["AI Search"][torque_question] = eval_row(torque_question, search_torque_hits, search_torque_answer)
    show_rag_turn(search_torque_prompt, search_torque_answer, {
        "says 42 Nm": yes_no("42" in search_torque_answer, expected=True),
        "cites": ", ".join(cited_labels(search_torque_answer, search_torque_hits)) or "nothing"},
        to=model("chat"), hits=search_torque_hits)

### Case K · An agent with the Azure AI Search tool

**Goal:** let Foundry query your index, as file search queries its vector store. The agent gets an `AzureAISearchTool` that points at the project connection and the index:

```python
AzureAISearchTool(azure_ai_search=AzureAISearchToolResource(indexes=[AISearchIndexResource(
    project_connection_id=search_connection.id, index_name=INDEX_NAME,
    query_type=AzureAISearchQueryType.SIMPLE, top_k=3)]))
```

| Input | Value |
|---|---|
| question | `torque_question` |
| agent | `friday-archive-search`, one tool, one index (the tool takes exactly one) |
| `query_type` | `SIMPLE`: keyword search. The default, `vector_semantic_hybrid`, needs the semantic ranker, which this index does not configure |
| `tool_choice` | `"required"`, as in the Microsoft Learn sample, so the agent always searches |

**Expected:** 42 Nm ❓, and `url_citation` annotations (not `file_citation`) that link to `arc-reactor-maintenance.md` through the `url` field ❓. The check table lists the output item types: ❓ the type of the search call was not seen yet. ❓ How the tool picks the content and URL fields of an index is not documented in detail; if citations are missing, that is the place to look.

In [ ]:
SEARCH_AGENT_NAME = "friday-archive-search"

if search_ready():
    from azure.ai.projects.models import AzureAISearchTool, AzureAISearchToolResource, AISearchIndexResource, AzureAISearchQueryType
    search_connection = project().connections.get(cfg("STARK_SEARCH_CONNECTION"))
    search_agent = project().agents.create_version(agent_name=SEARCH_AGENT_NAME, definition=PromptAgentDefinition(
        model=model("chat"), instructions=ARCHIVE_INSTRUCTIONS + " Always cite the sources you used.",
        tools=[AzureAISearchTool(azure_ai_search=AzureAISearchToolResource(indexes=[AISearchIndexResource(
            project_connection_id=search_connection.id, index_name=INDEX_NAME,
            query_type=AzureAISearchQueryType.SIMPLE, top_k=3)]))]))
    to_clean["agents"].append(SEARCH_AGENT_NAME)

    search_agent_reply = project_openai.responses.create(input=torque_question, extra_body=agent_ref(search_agent),
                                                         tool_choice="required")
    show_rag_turn(torque_question, answer_text(search_agent_reply), {
        "says 42 Nm": yes_no("42" in final_message(search_agent_reply), expected=True),
        "output items": ", ".join(f"`{name}`" for name in tool_calls(search_agent_reply)) or "none",
        "cited sources": ", ".join(cited_sources(search_agent_reply)) or "none"},
        to=SEARCH_AGENT_NAME, cites=citations(search_agent_reply))

## 5 · Score it: groundedness
Reading answers does not scale. Chapter 10 scored answers with evaluators; RAG adds evaluators that also read the **retrieved context**. Each case above stored a row with `query`, `context` and `response` in `rag_rows`.

| Evaluator | Reads | Question it answers |
|---|---|---|
| `builtin.groundedness` | `context`, `response`, `query` | does the answer say only what the context supports? |
| `builtin.relevance` | `query`, `response` | does the answer address the question? |
| `builtin.retrieval` | `query`, `context` | did retrieval bring the right context? |

Each is judged by `STARK_JUDGE_MODEL` on a scale of 1 to 5 and passes from 3.

### Step 1 · Define the evaluation

**Goal:** one evaluation for all variants. `sample=False`, because the answers are already in the rows (as in chapter 10, section 8).

**Expected:** an evaluation id. Nothing is scored yet.

In [ ]:
rag_schema = item_schema({"query": {"type": "string"}, "context": {"type": "string"}, "response": {"type": "string"}},
                         required=["query", "context", "response"], sample=False)
rag_evaluation = project_openai.evals.create(name="stark-rag-groundedness", data_source_config=rag_schema, testing_criteria=[
    criterion("groundedness", "groundedness", {"query": "{{item.query}}", "context": "{{item.context}}", "response": "{{item.response}}"}),
    criterion("relevance", "relevance", {"query": "{{item.query}}", "response": "{{item.response}}"}),
    criterion("retrieval", "retrieval", {"query": "{{item.query}}", "context": "{{item.context}}"}),
])
to_clean["evals"].append(rag_evaluation.id)
print("evaluation:", rag_evaluation.id)

### Step 2 · An answer you know is wrong

**Goal:** check the judge before you trust it. One hand-made row: the real question, the real context from step 3 of section 2 (it says 42 Nm), and an answer that says 90 Nm.

| Field | Value |
|---|---|
| `query` | `torque_question` |
| `context` | `torque_hits`, the three chunks |
| `response` | "The housing bolts are tightened to 90 Nm." |

`upload_rows()` ([stark_rag.py](stark_rag.py)) uploads the rows as the dataset `stark-rag-made-up-answer`; the run reads it with the `jsonl` data source from chapter 10.

**Expected:** `groundedness` fails with a low score: the context says 42 Nm ❓ (a judge score, but this row leaves little room). `relevance` may pass, because the answer does address the question. `retrieval` passes ❓: the context is the right one.

In [ ]:
made_up_row = eval_row(torque_question, torque_hits, "The housing bolts are tightened to 90 Nm.")
made_up_dataset = upload_rows("made-up answer", [made_up_row])
to_clean["datasets"].append(made_up_dataset.name)

made_up_run = project_openai.evals.runs.create(eval_id=rag_evaluation.id, name="made-up answer",
                                               data_source={"type": "jsonl", "source": {"type": "file_id", "id": made_up_dataset.id}})
made_up_run = wait_eval_run(rag_evaluation.id, made_up_run.id, project_openai)
show_scores(rag_evaluation.id, made_up_run.id, project_openai, reasons="all")

### Step 3 · Score every variant

**Goal:** one run per variant, over the rows its cases stored.

| Variant | Rows from |
|---|---|
| by hand | cases B, C and D step 2 |
| file search | cases F, G and H |
| AI Search | case J, only when section 4 ran |

A variant with no rows is skipped. Re-running a case replaces its row; re-running this cell uploads the rows again and adds new runs.

**Expected:** ❓ judge scores. The rows to read first are the Wi-Fi ones: an answer of "That is not in the archive." is grounded, while `retrieval` should fail there, because no context can answer that question. If file search returned no results in section 3 (the `include` ❓), its context is "(nothing retrieved)" and its `groundedness` and `retrieval` scores mean nothing.

In [ ]:
variant_runs = {}
for variant, rows in rag_rows.items():
    if not rows:
        print(f"skip {variant}: no rows")
        continue
    variant_dataset = upload_rows(variant, list(rows.values()))
    to_clean["datasets"].append(variant_dataset.name)
    run = project_openai.evals.runs.create(eval_id=rag_evaluation.id, name=variant,
                                           data_source={"type": "jsonl", "source": {"type": "file_id", "id": variant_dataset.id}})
    variant_runs[variant] = wait_eval_run(rag_evaluation.id, run.id, project_openai)
    print(f"--- {variant}")
    show_scores(rag_evaluation.id, run.id, project_openai)

### Step 4 · Compare

**Goal:** the pass rate per evaluator, one column per variant, with `show_pass_rates()` ([stark_evals.py](stark_evals.py)).

**Expected:** ❓ a table like the one below. The columns hold three, three and one row, so one answer moves a rate by 33 or 100 points: read the reasons in step 3 before you compare percentages.

| evaluator | by hand | file search | AI Search |
|---|---|---|---|
| groundedness | ... | ... | ... |
| relevance | ... | ... | ... |
| retrieval | ... | ... | ... |

In [ ]:
show_pass_rates(variant_runs)

## 👀 In the portal
- **Build → Agents**: `friday-archive-files` with its file search tool and the vector store `stark-archive`; `friday-archive-search` with the Azure AI Search tool (when section 4 ran). Ask the four questions in the **Playground**.
- **Evaluation → `stark-rag-groundedness`**: the runs `made-up answer`, `by hand`, `file search` and `AI Search`. Open a run for the judge's reasons per row.
- **Azure portal → your search service → Indexes → `stark-archive` → Search explorer** (when section 4 ran): run a keyword query against the 25 documents.

Look before you run the clean-up: it deletes all of them.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up
Deletes the evaluation, the row datasets, the vector store with its five uploaded files (`delete_vector_store` in [stark_helpers.py](stark_helpers.py)), both agents and the search index. The search **service** stays and keeps costing: remove it with `azd down`, or delete it in the Azure portal.

In [ ]:
for evaluation_id in to_clean["evals"]:
    try: project_openai.evals.delete(evaluation_id)
    except Exception as exc: print("skip eval", evaluation_id, type(exc).__name__)
for name in dict.fromkeys(to_clean["datasets"]):
    delete_dataset(name)
for store_id in to_clean["vector_stores"]:
    delete_vector_store(store_id, project_openai)
for name in dict.fromkeys(to_clean["agents"]):
    delete_agent(name)
for name in dict.fromkeys(to_clean["indexes"]):
    search_index_client().delete_index(name)
    print("deleted index", name)

## 🎯 Recap
| Step | By hand | File search | Azure AI Search |
|---|---|---|---|
| Index | `split_by_heading`, `embeddings.create` | `vector_stores.create`, `files.upload_and_poll` | `create_or_update_index`, `upload_documents` |
| Retrieve | cosine similarity, top-k | the tool, `include=["file_search_call.results"]` to see it | `search(search_text=..., vector_queries=[...])` |
| Answer | `grounded_prompt` + `responses.create(model=...)` | agent with `FileSearchTool` | your prompt, or an agent with `AzureAISearchTool` |
| Cite | `[n]` markers you ask for | `file_citation` | `url_citation` |
| Score | `builtin.groundedness`, `builtin.relevance`, `builtin.retrieval` over `query`, `context`, `response` | | |

- Retrieval always returns something. The rule "say when it is not there" and a groundedness check keep the answer honest.
- Chunking and `k` decide what the model can see. Change one at a time and look at the hits before you look at the answer.
- Start with file search. Move to your own index when you need control over chunks, fields, filters or the query type.

> 💡 Next: [12 · Conversation and trace evaluations](12_conversation_and_trace_evals.ipynb) 🧪 – judge whole conversations and live traffic.